# Deploy the VSS Orchestrator MCP and connect it to OpenClaw

Run this notebook on the same host that contains the VSS repo and will run the VSS orchestrator MCP server.

This notebook covers four things:

1. Pre-requisites and pre-steps.
2. Preparing the `services/agent/` Python environment.
3. Starting the VSS Orchestrator MCP server from `services/agent/src/vss_agents/orchestrator/`.
4. Updating the NemoClaw/OpenClaw sandbox policy so the OpenClaw sandbox created by `init_nemoclaw_vss.ipynb` can call the MCP endpoint.

It also includes exact examples for using the OpenClaw sandbox to initialize the MCP session and invoke the orchestrator tools.

**Important:** run `scripts/init_nemoclaw_vss.ipynb` first if OpenClaw/NemoClaw is not already configured on this instance.


## 1. Pre-requisites and pre-steps

Before you start this notebook, make sure these are already in place:

- The repo exists on this host, typically at `/home/ubuntu/video-search-and-summarization`.
- `scripts/init_nemoclaw_vss.ipynb` has already been run successfully, or you already have a working OpenClaw/NemoClaw sandbox.
- Docker and Docker Compose are installed and working.
- `openshell` is installed and can access the sandbox created by NemoClaw.
- `uv` and Python 3 are available on the host so `services/agent/` dependencies can be installed.
- If you plan to deploy local NIM-backed profiles through the orchestrator tools, make sure `NGC_CLI_API_KEY` is exported.
- If you plan to use remote NVIDIA endpoints in profile generation, make sure `NVIDIA_API_KEY` is exported.

Notes:

- The orchestrator MCP server runs on the host.
- OpenClaw reaches that host-side MCP server from inside the sandbox through `http://host.openshell.internal:<port>/mcp`.
- The default port used below is `9901`.


In [ ]:
import os
import shutil
from pathlib import Path

VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", "/home/ubuntu/video-search-and-summarization")).resolve()
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
ORCHESTRATOR_DIR = AGENT_DIR / "src" / "vss_agents" / "orchestrator"
MCP_CONFIG_PATH = ORCHESTRATOR_DIR / "vss_orchestrator_mcp_config.yml"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9901"))
MCP_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"
SANDBOX_MCP_URL = f"http://host.openshell.internal:{MCP_PORT}/mcp"

print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("ORCHESTRATOR_DIR:", ORCHESTRATOR_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("POLICY_PATH:", POLICY_PATH)
print("NEMOCLAW_SANDBOX_NAME:", NEMOCLAW_SANDBOX_NAME)
print("MCP_URL:", MCP_URL)
print("SANDBOX_MCP_URL:", SANDBOX_MCP_URL)
print()

checks = {
    "repo directory": VSS_REPO_DIR.is_dir(),
    "agent directory": AGENT_DIR.is_dir(),
    "orchestrator directory": ORCHESTRATOR_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
    "NemoClaw policy": POLICY_PATH.is_file(),
    "python3": shutil.which("python3") is not None,
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "openclaw": shutil.which("openclaw") is not None,
    "curl": shutil.which("curl") is not None,
}

for label, ok in checks.items():
    print(f"{'OK ' if ok else 'NO '} {label}")

print()
print("NGC_CLI_API_KEY set:", bool(os.environ.get("NGC_CLI_API_KEY")))
print("NVIDIA_API_KEY set:", bool(os.environ.get("NVIDIA_API_KEY")))


## 2. Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` first.

This runs `uv sync` from the agent directory.


In [ ]:
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

print("$ uv sync")
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), check=True)
print("Environment is ready.")


## 3. Review the orchestrator MCP configuration

The orchestrator server reads its runtime configuration from `services/agent/src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml`.

At minimum, confirm these paths are correct for this host before starting the server:

- `mdx_data_dir`
- `output_dir`
- `deployments_dir`
- `source_compose_yaml`
- `source_env`

If this repo is still under `/home/ubuntu/video-search-and-summarization`, the defaults should already be correct.


In [ ]:
import re

config_text = MCP_CONFIG_PATH.read_text()
for key in ["mdx_data_dir", "output_dir", "deployments_dir", "source_compose_yaml", "source_env"]:
    match = re.search(rf'^\s*{re.escape(key)}:\s*"?(.*?)"?\s*$', config_text, re.MULTILINE)
    print(f"{key}: {match.group(1) if match else 'NOT FOUND'}")

print()
print("Server start command:")
print(f"cd {AGENT_DIR} && uv run nat mcp serve --config_file src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml --port {MCP_PORT}")


## 4. Update and apply the NemoClaw/OpenClaw sandbox policy

This is the extra pre-step required so OpenClaw can talk to the orchestrator MCP running on the host.

The next cell does two things:

1. Ensures `assets/vss_nemoclaw_policy.yaml` contains a `host.openshell.internal:<port>` rule under `network_policies.vss-backend`.
2. Re-applies that policy to the current sandbox with `openshell policy set --wait`.

If you change the MCP port, re-run this cell so the policy stays in sync.


In [ ]:
import subprocess

policy_text = POLICY_PATH.read_text()
section_start = policy_text.find("  vss-backend:\n")
if section_start < 0:
    raise RuntimeError("Could not find the vss-backend policy section.")

binaries_idx = policy_text.find("    binaries:\n", section_start)
if binaries_idx < 0:
    raise RuntimeError("Could not find the binaries block for vss-backend.")

section_text = policy_text[section_start:binaries_idx]
port_line = f"        port: {MCP_PORT}\n"
if port_line not in section_text:
    insert_block = (
        "      - host: host.openshell.internal\n"
        f"        port: {MCP_PORT}\n"
        "        access: full\n"
        "        allowed_ips:\n"
        "          - 172.17.0.0/24\n"
    )
    policy_text = policy_text[:binaries_idx] + insert_block + policy_text[binaries_idx:]
    POLICY_PATH.write_text(policy_text)
    print(f"Updated {POLICY_PATH} with host.openshell.internal:{MCP_PORT}")
else:
    print(f"Policy already allows host.openshell.internal:{MCP_PORT}")

print()
print(f"$ openshell policy set --policy {POLICY_PATH} --wait {NEMOCLAW_SANDBOX_NAME}")
subprocess.run(
    ["openshell", "policy", "set", "--policy", str(POLICY_PATH), "--wait", NEMOCLAW_SANDBOX_NAME],
    check=True,
)

print()
print(f"$ openshell policy get {NEMOCLAW_SANDBOX_NAME}")
subprocess.run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME], check=True)


## 5. Start the VSS Orchestrator MCP server

This launches the server in the background, writes logs under `.orchestrator-artifacts/`, and stores the PID so later cells can reuse or stop it.

Default endpoint after startup: `http://127.0.0.1:9901/mcp`


In [ ]:
import os
import signal
import subprocess
import time

ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
PID_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.pid"

def pid_is_running(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

existing_pid = None
if PID_PATH.is_file():
    try:
        existing_pid = int(PID_PATH.read_text().strip())
    except ValueError:
        existing_pid = None

if existing_pid and pid_is_running(existing_pid):
    print(f"MCP server is already running with PID {existing_pid}")
else:
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    log_handle = LOG_PATH.open("a")
    process = subprocess.Popen(
        [
            "uv",
            "run",
            "nat",
            "mcp",
            "serve",
            "--config_file",
            "src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml",
            "--port",
            str(MCP_PORT),
        ],
        cwd=str(AGENT_DIR),
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
    )
    PID_PATH.write_text(str(process.pid))
    time.sleep(5)
    print(f"Started MCP server with PID {process.pid}")

print("PID file:", PID_PATH)
print("Log file:", LOG_PATH)
print()
if LOG_PATH.is_file():
    log_text = LOG_PATH.read_text()
    print(log_text[-4000:] if log_text else "Log file is empty.")


In [ ]:
import json
import urllib.request

def _parse_mcp_response_body(body: str):
    data_lines = [line[len("data:") :].strip() for line in body.splitlines() if line.startswith("data:")]
    if data_lines:
        return json.loads(data_lines[-1])
    return json.loads(body)

def _mcp_post(method: str, params: dict | None = None, *, session_id: str | None = None, request_id: int = 1):
    payload = json.dumps(
        {
            "jsonrpc": "2.0",
            "method": method,
            "params": params or {},
            "id": request_id,
        }
    ).encode("utf-8")
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if session_id:
        headers["mcp-session-id"] = session_id
    req = urllib.request.Request(MCP_URL, data=payload, headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=30) as resp:
        body = resp.read().decode("utf-8")
        return resp.headers, _parse_mcp_response_body(body)

initialize_headers, initialize_body = _mcp_post(
    "initialize",
    {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "notebook", "version": "1.0"},
    },
    request_id=0,
)
session_id = initialize_headers.get("mcp-session-id")
if not session_id:
    raise RuntimeError(f"MCP initialize succeeded but no mcp-session-id header was returned: {initialize_body}")

_, tools_body = _mcp_post("tools/list", {}, session_id=session_id, request_id=1)
tool_names = [tool["name"] for tool in tools_body.get("result", {}).get("tools", [])]
print("Tool names:")
for name in tool_names:
    print(" -", name)

print()
_, profiles_body = _mcp_post(
    "tools/call",
    {"name": "vss_orchestrator__docker_profiles", "arguments": {}},
    session_id=session_id,
    request_id=2,
)
print("docker_profiles response:")
print(json.dumps(profiles_body, indent=2))


## 6. Use OpenClaw to initialize and call the orchestrator MCP tools

After `init_nemoclaw_vss.ipynb` finishes and the policy-update cell above succeeds, the OpenClaw sandbox can reach the orchestrator MCP at:

- `http://host.openshell.internal:9901/mcp`

Use the following pattern from the OpenClaw sandbox for every MCP interaction.

### A. Minimal connectivity test from inside the OpenClaw sandbox

```bash
# Step 1: initialize and capture the session ID from the response header
SESSION_ID=$(curl -si -X POST http://host.openshell.internal:9901/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"openclaw","version":"1.0"}},"id":0}' \
  | awk 'BEGIN{IGNORECASE=1}/^mcp-session-id:/{print $2}' | tr -d '\r')

# Step 2: list tools using that session ID
curl -s -X POST http://host.openshell.internal:9901/mcp \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "mcp-session-id: $SESSION_ID" \
  -d '{"jsonrpc":"2.0","method":"tools/list","params":{},"id":1}' \
  | sed -n 's/^data: //p'
```

### B. OpenClaw prompt examples for every orchestrator MCP tool

Use these prompts in the OpenClaw UI that was configured by `init_nemoclaw_vss.ipynb`.

1. Discover the tool list:
   - `Use shell access in the sandbox to initialize an MCP session against http://host.openshell.internal:9901/mcp, then call tools/list and summarize the available orchestrator tools.`

2. `docker_profiles`:
   - `Call vss_orchestrator__docker_profiles and tell me which deployment profiles are supported.`

3. `docker_prereqs`:
   - `Call vss_orchestrator__docker_prereqs and summarize any failed prerequisite checks.`

4. `docker_generate`:
   - `Call vss_orchestrator__docker_generate for profile base with env_overrides ["HARDWARE_PROFILE=RTXPRO6000BW", "HOST_IP=<host-ip>"] and return the docker_compose_id.`
   - If you are generating a local-model deployment, include your NGC key in the tool input.
   - If you are generating a remote-endpoint deployment, include your NVIDIA API key in the tool input.

5. `docker_read`:
   - `Read the generated artifacts for docker_compose_id <id> with vss_orchestrator__docker_read and summarize the resolved env and compose file.`

6. `docker_up`:
   - `Start deployment for docker_compose_id <id> with vss_orchestrator__docker_up and tell me the returned docker_compose_ops_id.`

7. `docker_status`:
   - `Poll vss_orchestrator__docker_status for docker_compose_ops_id <ops-id> until the compose operation completes, and summarize the current log excerpt.`

8. `docker_list`:
   - `Call vss_orchestrator__docker_list with all_containers=true and list the container names.`

9. `docker_logs`:
   - `Call vss_orchestrator__docker_logs for container_name <name> with tail=200 and summarize the important log lines.`

10. `docker_down`:
   - `Tear down docker_compose_id <id> with vss_orchestrator__docker_down, then keep polling vss_orchestrator__docker_status until the shutdown finishes.`

### C. Recommended operating order from OpenClaw

A typical OpenClaw flow is:

1. `docker_profiles`
2. `docker_prereqs`
3. `docker_generate`
4. `docker_read`
5. `docker_up`
6. `docker_status`
7. `docker_list`
8. `docker_logs`
9. `docker_down` when you want to stop the deployment

If OpenClaw cannot reach the endpoint, re-run the policy-update cell above and make sure the MCP server is still listening on the same port.


In [ ]:
import os
import signal

if 'PID_PATH' not in globals():
    PID_PATH = VSS_REPO_DIR / '.orchestrator-artifacts' / 'vss_orchestrator_mcp.pid'

if PID_PATH.is_file():
    pid = int(PID_PATH.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to MCP server PID {pid}")
    except ProcessLookupError:
        print(f"Process {pid} is no longer running")
else:
    print("No PID file found. Nothing to stop.")
